# LangGraph ReAct Agent 示範

ReAct 代表 **Reasoning + Acting**。Agent 會先判斷需要哪些資訊，選擇工具執行，再根據工具結果繼續判斷，直到能產生最終答案。

本例的任務是：取得一段隨機的《Game of Thrones》名言，再搜尋說出該名言之角色的演員。

## 1. Import libraries

In [ ]:
import os
from typing import Dict

import requests
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from tavily import TavilyClient

## 2. 建立工具（Action）

第一個工具呼叫公開 API 取得隨機名言，第二個工具使用 Tavily 搜尋網路。請先在系統環境變數設定 `OPENAI_API_KEY` 與 `TAVILY_API_KEY`。

In [ ]:
@tool
def random_got_quote() -> Dict:
    """取得一段隨機的 Game of Thrones 名言及說出名言的角色。"""
    url = "https://api.gameofthronesquotes.xyz/v1/random"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()

In [ ]:
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def web_search(question: str) -> Dict:
    """搜尋網路以回答問題。"""
    return tavily_client.search(question)

工具可以單獨測試；以下兩格不是 ReAct 迴圈的必要部分。

In [ ]:
random_got_quote.invoke({})

In [ ]:
web_search.invoke({"question": "Who played Cersei Lannister in Game of Thrones?"})

## 3. 將工具提供給 LLM（Reasoning）

`bind_tools()` 會把工具名稱、說明與參數格式提供給模型。模型可以選擇直接回答，也可以透過 `tool_calls` 要求執行工具。

In [ ]:
tools = [random_got_quote, web_search]

llm = ChatOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    temperature=0.0,
)

llm_with_tools = llm.bind_tools(tools)

## 4. 建立 Agent 與 Router

- `agent`：讓 LLM 根據目前所有訊息進行判斷。
- `router`：若 LLM 產生 `tool_calls`，前往工具節點；否則結束並輸出答案。
- `ToolNode`：執行工具，並將結果轉成 `ToolMessage`（Observation）。

In [ ]:
def agent(state: MessagesState):
    ai_message = llm_with_tools.invoke(state["messages"])
    return {"messages": [ai_message]}


def router(state: MessagesState):
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END

## 5. 建立 ReAct Graph

最重要的是 `tools → agent` 這條回邊。工具執行完畢後，LLM 會觀察結果並再次判斷；只要仍有 `tool_calls`，循環就會繼續。

In [ ]:
workflow = StateGraph(MessagesState)

workflow.add_node("agent", agent)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    source="agent",
    path=router,
    path_map=["tools", END],
)
workflow.add_edge("tools", "agent")

graph = workflow.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## 6. 執行 ReAct Agent

我們只描述目標，不指定工具的呼叫順序。Agent 應先取得名言，再根據角色搜尋演員，最後整理答案。

In [ ]:
messages = [
    SystemMessage(
        content=(
            "You are a Game of Thrones research agent. "
            "When the user asks for a random quote, first use the quote tool. "
            "Then search the web for the actor or actress who played that character. "
            "Your final answer must include the quote, character, and performer."
        )
    ),
    HumanMessage(content="Give me a random Game of Thrones quote."),
]

result = graph.invoke({"messages": messages})

## 7. 觀察 ReAct 軌跡

以下訊息通常會依序包含：HumanMessage、帶有 `tool_calls` 的 AIMessage、工具回傳的 ToolMessage，以及最終 AIMessage。模型的私有思考不會顯示，但工具選擇與觀察結果可以從訊息中檢查。

In [ ]:
for index, message in enumerate(result["messages"], start=1):
    print(f"\n--- Step {index}: {message.__class__.__name__} ---")
    message.pretty_print()

In [ ]:
print("Final answer:")
print(result["messages"][-1].content)

## ReAct 對照

| ReAct 概念 | 本例中的實作 |
|---|---|
| Reason | `agent` 節點中的 LLM 判斷 |
| Action | `AIMessage.tool_calls` 指定工具與參數 |
| Observation | `ToolNode` 執行後產生的 `ToolMessage` |
| Loop | `tools → agent` 回邊 |
| Final Answer | 沒有新的 `tool_calls`，由 router 前往 `END` |

> 本例沒有加入 checkpointer，因為重點是單次 ReAct 任務。若要在多次 `graph.invoke()` 之間延續對話，再加入 `MemorySaver` 與 `thread_id`。